# **varlap to rust WIP**

## **Project Outline**

**Milestone 1**

- Implement a simple read first algorithm to make sure the concept works

**Milestone 2**

- Implement multithreading

**Milestone 3**

- Optimize program/make sure there’s even loading for each thread

**Milestone 4**

- Consider how to deal with overlapping in paired end sequencing reads
- Consider other statistics that may be useful with the information we have
- Maybe a feature that can save where you have progressed to if there is an error?

NOTES:

- Compare reads aligned in Bam in program to IGV (Note: default settings in IGV may not get all reads)
- Check how samtools/htslib gets the reads in a given region (default settings may skip low quality reads, or there may be a limit on how many reads are returned)

### **Setup notes**

To setup rust with jupyter:
- https://ratulmaharaj.com/posts/interactive-rust-with-jupyter-notebooks/

To setup varlap on Ubuntu with global Python 3.12:
- Error: `ERROR: Failed to build 'pysam' when getting requirements to build wheel`

```
$ git clone https://github.com/bjpop/varlap
$ cd varlap
$ sudo apt update
$ sudo apt install python3.7 python3.7-venv python3.7-distutils
$ python3.7 -m venv varlap_dev
$ source varlap_dev/bin/activate
$ pip install -U /path/to/varlap
```

Software used so far:

- WSL version: 2.6.3.0
- Kernel version: 6.6.87.2-1
- WSLg version: 1.0.71
- MSRDC version: 1.2.6353
- Direct3D version: 1.611.1-81528511
- DXCore version: 10.0.26100.1-240331-1435.ge-release
- Windows version: 10.0.26200.8037
- mamba version 2.2.3
- mamba install -c bioconda igv -y
- Ubuntu 24.04.3 LTS

### **Creating the structure for each variant:**
- In the original varlap, where each variant is looped over, variants are in a generator object (yield) to save memory
- However for our read first approach, we need to create objects/structs for all variants at the start and store all in the memory (Since we can’t be certain that for a given region, the first read in the bam file does not span the entire region)
- Therefore need to minimize what info we store in each variant structure
- Varlap stores: {"chrom": chrom, "pos": pos, "ref": ref, "alt": alt}, counts of bases, variant type, and associated statistics
- Skip statistics for now

#### Variant

Initialise it with what we know for chrom/pos/ref/alt (Note: ref can't be used so we use refr instead; maybe change to something else?)
- **NOTE: u32 vs u64: realistically is there any performance upgrade?**

In [2]:
#[derive(Debug, Clone)]
struct VariantReader {
	chrom: String,
	pos: u64,
	refr: String,
	alt: String,
}

#### Base Counts

For base counts we can use a struct; we also implement a counter for the structure so that every time we match increment the base count

In [3]:
#[derive(Debug, Clone, Default)]
struct BaseCounts {
    a: u32,
    c: u32,
    g: u32,
    t: u32,
    n: u32,
}


impl BaseCounts {
    fn increment(&mut self, base: char) {
        match base {
            'A' => self.a += 1,
            'C' => self.c += 1,
            'G' => self.g += 1,
            'T' => self.t += 1,
            'N' => self.n += 1,
            _ => eprintln!("Warning: Base does not match: {}", base),
        }
    }
}

#### Variant Types

The python code:
```
def get_var_type(ref, alt):
    if len(ref) == 1 and len(alt) == 1:
        return "SNV"
    elif len(ref) > len(alt):
        return "DEL"
    elif len(alt) > len(ref):
        return "INS"
    else:
        logging.warning(f"Cannot determine the type of variant with ref: {ref} and alt: {alt}")
        return "UNKNOWN"
```

For variant types we can use an enum to define each type (use enum instead of struct as types are mutually exclusive) instead of storing each type as a string like the python code

We also implement a function that converts the type into a string for later; use `&'static str` as we know they are fixed size literals; avoids allocating new string to each variant in heap

In [4]:
#[derive(Debug, Clone, Copy)]
enum VarType {
    Snv,
    Del,
    Ins,
    Unknown,
}

impl VarType {
    fn as_str(&self) -> &'static str {
        match self {
            VarType::Snv => "SNV",
            VarType::Del => "DEL",
            VarType::Ins => "INS",
            VarType::Unknown => "UNKNOWN",            
        }
    }
}



We also need to implement a function that gets the variant type from the ref and alt alleles of a variant

In [5]:
fn get_var_type(refr: &str, alt: & str) -> VarType {
    if refr.len() == 1 && alt.len() == 1 {
        VarType::Snv
    } else if refr.len() > alt.len() {
        VarType::Del
    } else if refr.len() < alt.len() {
        VarType::Ins
    } else {
        eprintln!(
            "Warning: Cannot determine the type of variant with ref: {} and alt: {}",
            refr, alt
        );
        VarType::Unknown
    }
}

Update the variant structure with what we have just implemented:

In [6]:
#[derive(Debug, Clone)]
struct Variant {
	chrom: String,
	pos: u64,
	refr: String,
	alt: String,
	vartype: VarType,
	counts: BaseCounts,
}

### **VCF Reader**
Now we need to implement a function that reads a VCF (implement csv/tsv later) and creates an vector of Variant structures for each variant
- Variants need to be in a queue as we want to drop them from memory once the start of the read position is > than the variant position

Python code:
```
def vcf_reader(file):
    for line in file:
        if line.startswith('#'):
            continue
        fields = line.strip().split()
        # Technically VCF requires the first 8 fields to be defined, but we want to be as liberal
        # as possible in accepting inputs.
        if len(fields) >= 5:
            chrom, pos, _id, ref, alt = fields[:5]
            yield {"chrom": chrom, "pos": pos, "ref": ref, "alt": alt}
        else:
            logging.warning(f"Skipping input row: {line}")
```
and
```
    def get_variants(self):
        '''Read variants from input VCF file, yield one at a time'''
        for input_row in self.reader: 
            self.total_variants_in_input += 1
            if is_valid_input_row(input_row):
                this_ref = input_row["ref"]
                # allow possibly multiple alts in the same variant, split them into separate alleles
                alts = input_row["alt"].split(",")
                for this_alt in alts:
                    this_var_type = get_var_type(this_ref, this_alt)
                    if is_acceptable_variant(dict(input_row), self.varclass, this_var_type, this_ref, this_alt, self.max_indel_size):
                        output_row = copy(input_row)
                        output_row["alt"] = this_alt
                        output_row["pos"] = int(input_row["pos"])
                        output_row["vartype"] = this_var_type
                        self.num_variants_analysed += 1
                        yield output_row
            else:
                logging.warning(f"Skipping invalid input row: {dict(input_row)}")
```

In [7]:
use std::fs::File;
use std::io::{self, BufRead, BufReader};
use std::collections::VecDeque;
use std::error::Error;

fn vcf_reader(file_path: &str) -> Result<VecDeque<Variant>, Box<dyn Error>> {
    let file = File::open(file_path)?;
    let reader = BufReader::new(file);

	let mut variants = VecDeque::new();

    for line_result in reader.lines() {
        let line = line_result?;
        if line.starts_with("#") {
            continue;
            }
        
        let fields: Vec<&str> = line.split_whitespace().collect();
        
        if fields.len() >= 5 {
            let chrom = fields[0].to_string();

            let pos = match fields[1].parse::<u64>() {
                Ok(p) => p,
                Err(_) => {
                    eprintln!("Warning: invalid POS, skipping row: {}", line);
                    continue;
                }
            };

            let refr = fields[3].to_string();

            for alt in fields[4].split(',') {
                let vartype = get_var_type(&refr, &alt);

                variants.push_back(Variant {
                    chrom: chrom.clone(),
                    pos,
                    refr: refr.clone(),
                    alt: alt.to_string(),
                    vartype,
                    counts: BaseCounts::default(),
                });
            }
        } else {
            eprintln!("Warning: Skipping input row: {}", line);
        }
    }
			
	Ok(variants)
}

Notes:

- Need `let line = line_result?;` as line yields `Option<Result<String, std::io::Error>>`; `?` unwraps the `Ok(String)` case or returns an error if it cannot read the line
- We use `.to_string()` to convert the string slice in `fields` to an actual `String` with ownership; we then have to use `.clone()` when looping over all alts, otherwise the first chrom/refr String will get consumed in the first iteration
- `u32` type has the `Copy` trait and does not get consumed so we do not need to clone; using the match field converts to an actual `u64` and skips if there is an error
- We multiple alts in the same variant by separating them and creating different Variant structs; we assume that they are separated by `,`
- **Should we use `Arc<str>` for chrom globally and for the reference allele locally for each variant? How often are there multiallelic alts for this to be worth it? Or store chrom as an id integer?**
- 

We also need to create a function that gets the min and max positions of all the variants (i.e. the interval that all the variants span); we can then use this to get all reads in the bam file that overlap this region/interval
- If we assume that the vcf contains variants from only one chromosome, and that they are sorted we can use:

In [8]:
fn get_vcf_min_max(variants: &VecDeque<Variant>) -> Option<(String, u64, u64)> {
    let first = variants.front()?;
    let chrom = first.chrom.clone();

    let min_pos = first.pos;
    let max_pos = variants.back()?.pos;

    Some((chrom, min_pos, max_pos))
}

#### Lets test our functions so far using a small vcf

In [9]:
use std::fs;

let vcf_path = "/home/jch/git/rust-varlap/test_data/vars.vcf";

let data = fs::read_to_string(&vcf_path).expect("Should be able to read file");
println!("{}", data);

##fileformat=VCFv4.2
##contig=<ID=chr1,length=260>
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	100	.	AT	A	.	.	.
chr1	115	.	G	C	.	.	.
chr1	130	.	C	T,G	.	.	.
chr1	145	.	A	G	.	.	.
chr1	160	.	T	C	.	.	.
chr1	175	.	G	A	.	.	.
chr1	190	.	C	CT	.	.	.
chr1	200	.	T	G	.	.	.



In [10]:
let variants = vcf_reader(&vcf_path)?;

let (region_chrom, min_pos, max_pos) = 
    get_vcf_min_max(&variants).ok_or("Could not determine VCF min/max")?;

println!("Region Chromosome: {}, Min Pos: {}, Max Pos: {} \n", region_chrom, min_pos, max_pos);

println!("All Variants in the Variants VecDeque:");
for line in variants {
    println!("{:?}", line);
}

Region Chromosome: chr1, Min Pos: 100, Max Pos: 200 

All Variants in the Variants VecDeque:
Variant { chrom: "chr1", pos: 100, refr: "AT", alt: "A", vartype: Del, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 115, refr: "G", alt: "C", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 130, refr: "C", alt: "T", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 130, refr: "C", alt: "G", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 145, refr: "A", alt: "G", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 160, refr: "T", alt: "C", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 175, refr: "G", alt: "A", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 190, ref

()

#### **Base Count Statistics**

We can also create a new structure to store statistics calculated from the base stats after all bases have been counted (Or maybe don't even need a structure to store since it's a one time calculation)
- Need to calculate the depth, reference count, alt count, and alt_vaf (alt_count/depth)
```
fields = ["depth", "A", "T", "G", "C", "N", "ref", "alt", "alt vaf"]
    def as_list(self):
        total_depth = self.A + self.T + self.G + self.C + self.N
        ref_count = getattr(self, self.ref)
        alt_count = getattr(self, self.alt)
        if total_depth > 0:
            alt_vaf = alt_count / total_depth
        else:
            alt_vaf = ''
        return [total_depth, self.A, self.T, self.G, self.C, self.N, ref_count, alt_count, alt_vaf]
```

In [11]:
#[derive(Debug, Clone, Copy)]
struct BaseCountsStats {
    depth: u32,
    refr_count: u32,
    alt_count: u32,
    alt_vaf: f64,
}

We can then update the BaseCount struct and Variant struct to include functions which calculate these stats

In [12]:
impl BaseCounts {
    fn count_for_base(&self, base: char) -> u32 {
        match base {
            'A' => self.a,
            'C' => self.c,
            'G' => self.g,
            'T' => self.t,
            'N' => self.n,
            _ => 0,
        }
    }

    fn depth(&self) -> u32 {
        self.a + self.c + self.g + self.t + self.n
    }

    fn stats(&self, refr: char, alt: char) -> BaseCountsStats {
        let depth = self.depth();
        let refr_count = self.count_for_base(refr);
        let alt_count = self.count_for_base(alt);
        let alt_vaf = if depth > 0 {
            alt_count as f64 / depth as f64
        } else {
            0.0
        };

        BaseCountsStats {
            depth,
            refr_count,
            alt_count,
            alt_vaf,
        }
    }
}

In [13]:
impl Variant {
    fn base_counts_stats(&self) -> Option<BaseCountsStats> {
        let refr_char = self.refr.chars().next()?;
        let alt_char = self.alt.chars().next()?;
        Some(self.counts.stats(refr_char, alt_char))
    }
}

Some functions to test output of all info so far; lets make it match with how varlap is outputted so we can compare

In [14]:
fn print_header_row() -> Result<(), Box<dyn Error>> {
    println!("chrom\tpos\tref\talt\tvartype\tdepth\ta\tt\tg\tc\tn\trefr_count\talt_count\talt_vaf");
    Ok(())
}

fn print_variant_row(var: &Variant, base_stats: &BaseCountsStats) -> Result<(), Box<dyn Error>> {
    println!(
        "{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}",
        var.chrom,
        var.pos,
        var.refr,
        var.alt,
        var.vartype.as_str(),
        base_stats.depth,
        var.counts.a,
        var.counts.t,
        var.counts.g,
        var.counts.c,
        var.counts.n,
        base_stats.refr_count,
        base_stats.alt_count,
        base_stats.alt_vaf,
    );
    Ok(())
}

In [15]:
let variants = vcf_reader(&vcf_path)?;
print_header_row()?;
for variant in variants {
    let base_counts_stats = variant.base_counts_stats().ok_or("error")?;
    print_variant_row(&variant, &base_counts_stats)?;
}


chrom	pos	ref	alt	vartype	depth	a	t	g	c	n	refr_count	alt_count	alt_vaf
chr1	100	AT	A	DEL	0	0	0	0	0	0	0	0	0
chr1	115	G	C	SNV	0	0	0	0	0	0	0	0	0
chr1	130	C	T	SNV	0	0	0	0	0	0	0	0	0
chr1	130	C	G	SNV	0	0	0	0	0	0	0	0	0
chr1	145	A	G	SNV	0	0	0	0	0	0	0	0	0
chr1	160	T	C	SNV	0	0	0	0	0	0	0	0	0
chr1	175	G	A	SNV	0	0	0	0	0	0	0	0	0
chr1	190	C	CT	INS	0	0	0	0	0	0	0	0	0
chr1	200	T	G	SNV	0	0	0	0	0	0	0	0	0


()

### **BAM Reader**

First we need to get all reads within the min/max region of all variants
- To use dependencies in evcxr use 
`:dep <dependency>`
- Also need to install libclang if you do not have it already
`sudo apt install llvm-dev libclang-dev clang`

In [16]:
:dep rust-htslib

- For fetch(chrom, start, stop): start and stop are zero-based. start is inclusive, stop is exclusive
- VCF is one-based so (start-1), stop can be the same
- Iterate over all reads; get the start position and read sequence (for INDELS may need to decode sequence using as_bytes())
- Iterate over variants; we turn our variant into zero-based and get the read_end using .len(); then check whether variant is in these bounds; if yes, increment bases (later implement ReadFeatures); if not break as we assume variants are sorted

Note that this is only for SNVs; to apply to INDELS we need to work with CIGAR

In [17]:
use rust_htslib::bam::{Read, IndexedReader};

fn process_bam_region(
    variants: &mut VecDeque<Variant>,
    file_path: &str,
    region_chrom: &str, 
    min_pos: u64, 
    max_pos: u64,
) -> Result<(), Box<dyn std::error::Error>> {
    let mut bam_reader = IndexedReader::from_path(file_path)?;
    //let header = bam_reader.header().to_owned();

    bam_reader.fetch((region_chrom, min_pos - 1, max_pos))?;

    for read_result in bam_reader.rc_records() {
        let record = read_result?;

        // probably don't need this tid check since indexedreader already specifies reads from a given chromsome
        //let tid = record.tid();
        //let read_chrom = String::from_utf8(header.tid2name(tid as u32).to_vec())?;

        let read_start = record.pos() as u64;

        let seq = record.seq();

        for variant in &mut *variants {
            //if variant.chrom != read_chrom {
            //    break;
            //}
            
            let zero_based_pos = variant.pos - 1;
            let read_end = read_start + record.seq_len() as u64;

            if zero_based_pos >= read_start && zero_based_pos < read_end {
                let base = seq[(zero_based_pos - read_start) as usize] as char;
                variant.counts.increment(base);
            } else {
                break;
            }
        }
    }

    Ok(())    
}

Note: Was considering using: https://docs.rs/rust-htslib/latest/rust_htslib/bam/record/struct.Seq.html, but was unsure of use compare to []

Links to interesting example:
https://doc.rust-lang.org/nightly/core/ops/trait.Index.html#associatedtype.Output

Lets check our base counts after implementing the BAM reader for SNVs


In [18]:
let bam_path = "/home/jch/git/rust-varlap/test_data/reads.sorted.bam";

let mut variants = vcf_reader(&vcf_path)?;

process_bam_region(&mut variants, &bam_path, &region_chrom, min_pos, max_pos)?;

print_header_row()?;
for variant in &variants {
    let base_counts_stats = variant.base_counts_stats().ok_or("error")?;
    print_variant_row(&variant, &base_counts_stats)?;
}

chrom	pos	ref	alt	vartype	depth	a	t	g	c	n	refr_count	alt_count	alt_vaf
chr1	100	AT	A	DEL	7	0	7	0	0	0	0	0	0
chr1	115	G	C	SNV	6	0	0	6	0	0	6	0	0
chr1	130	C	T	SNV	4	0	0	0	4	0	4	0	0
chr1	130	C	G	SNV	4	0	0	0	4	0	4	0	0
chr1	145	A	G	SNV	1	1	0	0	0	0	1	0	0
chr1	160	T	C	SNV	0	0	0	0	0	0	0	0	0
chr1	175	G	A	SNV	0	0	0	0	0	0	0	0	0
chr1	190	C	CT	INS	0	0	0	0	0	0	0	0	0
chr1	200	T	G	SNV	0	0	0	0	0	0	0	0	0


()

Lets compare it to the varlap output we created using:

`$ varlap --varclass SNV --format VCF -- /home/jch/git/rust-varlap/test_data/reads.sorted.bam < /home/jch/git/rust-varlap/test_data/vars.vcf > variants.varlap.csv`

`$ column -s, -t < variants.varlap.csv | less -S`

In [19]:
:dep csv

In [20]:
use csv::Reader;

let varlap_csv = "/home/jch/git-fork/varlap/variants.varlap.csv";

let mut csv_reader = csv::ReaderBuilder::new()
        .has_headers(false)
        .from_path(varlap_csv)?;

for record in csv_reader.records() {
    let line = record?;
    println!("{:?}", line);
}


The type of the variable variants was redefined, so was lost.


StringRecord(["chrom", "pos", "ref", "alt", "vartype", "pos normalised", "sample", "reads.sorted.bam depth", "reads.sorted.bam A", "reads.sorted.bam T", "reads.sorted.bam G", "reads.sorted.bam C", "reads.sorted.bam N", "reads.sorted.bam ref", "reads.sorted.bam alt", "reads.sorted.bam alt vaf", "reads.sorted.bam ref avg NM", "reads.sorted.bam ref avg base qual", "reads.sorted.bam ref avg map qual", "reads.sorted.bam ref avg align len", "reads.sorted.bam ref avg clipped bases", "reads.sorted.bam ref avg indel bases", "reads.sorted.bam ref fwd strand", "reads.sorted.bam ref rev strand", "reads.sorted.bam ref supplementary", "reads.sorted.bam ref normalised read position", "reads.sorted.bam alt avg NM", "reads.sorted.bam alt avg base qual", "reads.sorted.bam alt avg map qual", "reads.sorted.bam alt avg align len", "reads.sorted.bam alt avg clipped bases", "reads.sorted.bam alt avg indel bases", "reads.sorted.bam alt fwd strand", "reads.sorted.bam alt rev strand", "reads.sorted.bam alt supp

()

Why the difference? Because we haven't implemented the dropping of the queue after a variant has been proccessed. This means that the line `if zero_based_pos >= read_start && zero_based_pos < read_end ` may trigger a break for variants that are later in the queue. We are also getting all variant types instead of just SNVs.

Let's implement the dropping of the queue now. For now we'll use print statements instead of writing to a csv as it is easier to read

In [21]:
fn process_bam_region(
    variants: &mut VecDeque<Variant>,
    bam_path: &str,
    region_chrom: &str, 
    min_pos: u64, 
    max_pos: u64,
) -> Result<(), Box<dyn std::error::Error>> {
    let mut bam_reader = IndexedReader::from_path(bam_path)?;
    //let header = bam_reader.header().to_owned();

    bam_reader.fetch((region_chrom, min_pos - 1, max_pos))?;

    print_header_row()?;

    println!("Variant queue length before loop: {}", variants.len());

    for read_result in bam_reader.rc_records() {
        let record = read_result?;

        //let tid = record.tid();
        //let read_chrom = String::from_utf8(header.tid2name(tid as u32).to_vec())?;

        let read_start = record.pos() as u64;

        loop {
            let should_pop = match variants.front() {
                // Use read_start + 1 because comparing to vcf one-based pos
                Some(var) => (read_start + 1) > var.pos,
                None => false,
            };

            if should_pop {
                if let Some(var) = variants.pop_front() {
                    let base_counts_stats = var.base_counts_stats().ok_or("error")?;
                    print_variant_row(&var, &base_counts_stats)?;
                }

            println!("Variant queue length after loop activates: {}", variants.len());
            } else {
                break;
            }
        }

        let seq = record.seq();

        for var in &mut *variants {
            //if variant.chrom != read_chrom {
            //    break;
            //}
            
            let zero_based_pos = var.pos - 1;
            let read_end = read_start + record.seq_len() as u64;

            if zero_based_pos >= read_start && zero_based_pos < read_end {
                let base = seq[(zero_based_pos - read_start) as usize] as char;
                var.counts.increment(base);
            } else {
                break;
            }
        }
    }

    // After going through all reads we still need to process the variants left
    while let Some(var) = variants.pop_front() {
        let base_counts_stats = var.base_counts_stats().ok_or("error")?;
        print_variant_row(&var, &base_counts_stats)?;
    }

    println!("Variant queue length at end: {}", variants.len());

    Ok(())    
}

And for variant class selection (should be a user input later); we need to match SNV and INDEL (NOTE: what to do with unknowns?)

Can do further checking later (associated python code): 

https://github.com/bjpop/varlap/blob/f4a48c801187d0427caba1010dbfc623eb0d49b4/varlap/varlap.py#L216

https://github.com/bjpop/varlap/blob/f4a48c801187d0427caba1010dbfc623eb0d49b4/varlap/varlap.py#L299

In [22]:
fn varclass_matches(varclass: &str, vartype: &VarType) -> bool {
    match varclass.to_ascii_uppercase().as_str() {
        "SNV" => matches!(vartype, VarType::Snv | VarType::Unknown),
        "INDEL" => matches!(vartype, VarType::Ins | VarType::Del | VarType::Unknown),
        _ => false,
    }
}

Lets add it to our vcf_reader function

In [23]:
fn vcf_reader(file_path: &str, varclass: &str) -> Result<VecDeque<Variant>, Box<dyn Error>> {
    let file = File::open(file_path)?;
    let reader = BufReader::new(file);

	let mut variants = VecDeque::new();

    for line_result in reader.lines() {
        let line = line_result?;
        if line.starts_with("#") {
            continue;
            }
        
        let fields: Vec<&str> = line.split_whitespace().collect();
        
        if fields.len() >= 5 {
            let chrom = fields[0].to_string();

            let pos = match fields[1].parse::<u64>() {
                Ok(p) => p,
                Err(_) => {
                    eprintln!("Warning: invalid POS, skipping row: {}", line);
                    continue;
                }
            };

            let refr = fields[3].to_string();

            for alt in fields[4].split(',') {
                let vartype = get_var_type(&refr, alt);

                if !varclass_matches(varclass, &vartype) {
                    continue;
                }

                variants.push_back(Variant {
                    chrom: chrom.clone(),
                    pos,
                    refr: refr.clone(),
                    alt: alt.to_string(),
                    vartype,
                    counts: BaseCounts::default(),
                });
            }
        } else {
            eprintln!("Warning: Skipping input row: {}", line);
        }
    }
			
	Ok(variants)
}

Ok, lets run everything again and check our outputs

In [24]:
let varclass = String::from("SNV");

let mut variants = vcf_reader(&vcf_path, &varclass)?;

process_bam_region(&mut variants, &bam_path, &region_chrom, min_pos, max_pos)?;


chrom	pos	ref	alt	vartype	depth	a	t	g	c	n	refr_count	alt_count	alt_vaf
Variant queue length before loop: 7
chr1	115	G	C	SNV	9	0	0	8	1	0	8	1	0.1111111111111111
Variant queue length after loop activates: 6
chr1	130	C	T	SNV	10	0	0	0	10	0	10	0	0
Variant queue length after loop activates: 5
chr1	130	C	G	SNV	10	0	0	0	10	0	10	0	0
Variant queue length after loop activates: 4
chr1	145	A	G	SNV	10	10	0	0	0	0	10	0	0
Variant queue length after loop activates: 3
chr1	160	T	C	SNV	10	0	10	0	0	0	10	0	0
Variant queue length after loop activates: 2
chr1	175	G	A	SNV	10	0	0	10	0	0	10	0	0
Variant queue length after loop activates: 1
chr1	200	T	G	SNV	7	0	7	0	0	0	7	0	0
Variant queue length at end: 0


#### Chromosome position fraction

Lets create the function that gets the variant position as a fraction of the chromsome length:
- We can first write a function that gets the chromosome reference sequence length from the header
- We can then add an implementation function to Variant (as it already stores the variant position) to calculate the normalized variant position

In [ ]:
fn get_ref_len(
    bam_reader: &IndexedReader,
    chrom: &str,
) -> Result<u64, Box<dyn std::error::Error>> {
    let header = bam_reader.header();

    for tid in 0..header.target_count() {
        let name = std::str::from_utf8(header.tid2name(tid))?;
        if name == chrom {
            return header
                .target_len(tid)
                .ok_or_else(|| format!("Reference '{}' found, but has no length", chrom).into());
        }
    }

    Err(format!("Could not find reference '{}' in BAM header", chrom).into())
}

impl Variant {
    fn get_pos_fraction(&self, ref_seq_len: u64) -> f64 {
        self.pos as f64/ ref_seq_len as f64
    }
}

Then to use it we just assign to a variable

In [ ]:
let ref_seq_len = get_ref_len(&bam_reader, &region_chrom)?;

let pos_fraction = var.get_pos_fraction(ref_seq_len);

#### Writing to CSV

NOTE: Will probably change to serde: serialize later

In [ ]:
fn write_header_row(writer: &mut Writer<File>) -> Result<(), Box<dyn Error>> {
    writer.write_record(&[
        "chrom",
        "pos",
        "ref",
        "alt",
        "vartype",
        "pos_normalised",
        "depth",
        "A",
        "T",
        "G",
        "C",
        "N",
        "ref_count",
        "alt_count",
        "alt_vaf",
    ])?;
    Ok(())
}

fn write_variant_row(
    writer: &mut Writer<File>,
    var: &Variant,
    base_stats: &BaseCountsStats,
    pos_fraction: f64,
) -> Result<(), Box<dyn Error>> {
    writer.write_record(&[
        &var.chrom,
        &var.pos.to_string(),
        &var.refr,
        &var.alt,
        &var.vartype.as_str().to_string(),
        &pos_fraction.to_string(),
        &base_stats.depth.to_string(),
        &var.counts.a.to_string(),
        &var.counts.t.to_string(),
        &var.counts.g.to_string(),
        &var.counts.c.to_string(),
        &var.counts.n.to_string(),
        &base_stats.refr_count.to_string(),
        &base_stats.alt_count.to_string(),
        &base_stats.alt_vaf.to_string(),
    ])?;
    Ok(())
}

Using the functions; also flushing the writer at the end just in case

In [ ]:
let mut csv_writer = Writer::from_path(csv_path)?;

write_header_row(&mut csv_writer)?;

write_variant_row(&mut csv_writer, &var, &base_counts_stats, pos_fraction)?;

csv_writer.flush()?;

### **Read Feature**

Lets now implement the reads features so that we can compare the timing to the original varlap

- htslib does not have CIGAR string > 10 in pysam pileup alignment, so we instead get the edit distance (NM) from the auxillary fields

In [ ]:
use rust_htslib::bam::record::{Aux, Cigar};
use std::rc::Rc;

#[derive(Debug, Clone, Default)]
struct ReadFeatures {
    nm: u32,
    base_qual: u32,
    map_qual: u32,
    align_len: u32,
    clipping: u32,
    indel: u32,
    forward_strand: u32,
    reverse_strand: u32,
    supplementary: u32,
    normalised_read_position: f64, 
    num_reads: u32,
}

impl ReadFeatures {
    fn count(
        &mut self,
        read: &Rc<Record>,
        query_pos: u64,
    ) {
        self.num_reads += 1;
        let query_len = read.seq_len();
        if query_len > 0 {
            self.normalised_read_position += query_pos as f64 / query_len as f64;
        }
        // Can maybe to Some(pos_qual) = read.qual().get() if want to return an Option for safety
        let pos_qual = read.qual()[query_pos as usize];
        self.base_qual += pos_qual as u32;

        self.align_len += query_len as u32;
        self.map_qual += read.mapq() as u32;

        for c in read.cigar().iter() {
            match *c {
                Cigar::Ins(len) | Cigar::Del(len) => self.indel += len as u32,
                Cigar::SoftClip(len) | Cigar::HardClip(len) => self.clipping += len as u32,
                _ => {}
            }
        }

        if let Ok(aux) = read.aux(b"NM") {
            match aux {
                Aux::I8(v)  => self.nm += v as u32,
                Aux::U8(v)  => self.nm += v as u32,
                Aux::I16(v) => self.nm += v as u32,
                Aux::U16(v) => self.nm += v as u32,
                Aux::I32(v) => self.nm += v as u32,
                Aux::U32(v) => self.nm += v as u32,
                _ => {}
            }
        }

        if read.is_reverse() {
            self.reverse_strand += 1;
        } else {
            self.forward_strand += 1;
        }
        if read.is_supplementary() {
            self.supplementary += 1;
        }

    }

    fn normalized(&self) -> NormalizedReadFeatures {
        if self.num_reads > 0 {
            let n = self.num_reads as f64;
            NormalizedReadFeatures {
                nm: Some(self.nm as f64 / n),
                base_qual: Some(self.base_qual as f64 / n),
                map_qual: Some(self.map_qual as f64 / n),
                align_len: Some(self.align_len as f64 / n),
                clipping: Some(self.clipping as f64 / n),
                indel: Some(self.indel as f64 / n),
                forward_strand: Some(self.forward_strand as f64 / n),
                reverse_strand: Some(self.reverse_strand as f64 / n),
                supplementary: Some(self.supplementary as f64 / n),
                normalised_read_position: Some(self.normalised_read_position / n),
            }
        } else {
            NormalizedReadFeatures::default()
        }
    }
}


In [ ]:
#[derive(Debug, Clone, Default)]
struct LocusFeaturesSNV {
    ref_read_features: ReadFeatures,
    alt_read_features: ReadFeatures,
    all_read_features: ReadFeatures,
}

impl LocusFeaturesSNV {
    fn count(&mut self, read: &Rc<Record>, base: char, refr: char, alt: char, query_pos: u64) {
        if base == refr {
            self.ref_read_features.count(&read, query_pos);
        } else if base == alt{
            self.alt_read_features.count(&read, query_pos);
        }
        self.all_read_features.count(&read, query_pos);
    }

    fn normalized_row(&self) -> NormalizedLocusFeaturesRow {
        let r = self.ref_read_features.normalized();
        let a = self.alt_read_features.normalized();
        let all = self.all_read_features.normalized();

        NormalizedLocusFeaturesRow {
            ref_nm: r.nm,
            ref_base_qual: r.base_qual,
            ref_map_qual: r.map_qual,
            ref_align_len: r.align_len,
            ref_clipping: r.clipping,
            ref_indel: r.indel,
            ref_forward_strand: r.forward_strand,
            ref_reverse_strand: r.reverse_strand,
            ref_supplementary: r.supplementary,
            ref_normalised_read_position: r.normalised_read_position,

            alt_nm: a.nm,
            alt_base_qual: a.base_qual,
            alt_map_qual: a.map_qual,
            alt_align_len: a.align_len,
            alt_clipping: a.clipping,
            alt_indel: a.indel,
            alt_forward_strand: a.forward_strand,
            alt_reverse_strand: a.reverse_strand,
            alt_supplementary: a.supplementary,
            alt_normalised_read_position: a.normalised_read_position,

            all_nm: all.nm,
            all_base_qual: all.base_qual,
            all_map_qual: all.map_qual,
            all_align_len: all.align_len,
            all_clipping: all.clipping,
            all_indel: all.indel,
            all_forward_strand: all.forward_strand,
            all_reverse_strand: all.reverse_strand,
            all_supplementary: all.supplementary,
            all_normalised_read_position: all.normalised_read_position,
        }
    }    
}

Update the Variant struct

- NOTE: Can probably move the basecounts into read features like original varlap

In [ ]:
#[derive(Debug, Clone)]
struct Variant {
	chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType,
	counts: BaseCounts,
    read_features: LocusFeaturesSNV,
}

For our csv writer, instead of concatenating vectors we can use the serialize of the 'Serde' crate instead

We use Option<> for the ReadFeatures output as we are using Some(T); when it is None, serde will write as an empty space like in the original varlap; otherwise, if we do not want this as an empty spce, we do not have to use Some(T) and can just return a f64 of 0.0

In [ ]:
use serde::Serialize;

#[derive(serde::Serialize)]
struct OutputRow<'a> {
    chrom: &'a str,
    pos: u64,
    #[serde(rename = "ref")]
    refr: &'a str,
    alt: &'a str,
    vartype: &'a str,
    pos_normalised: f64,
    depth: u32,

    count_a: u32,
    count_t: u32,
    count_g: u32,
    count_c: u32,
    count_n: u32,

    ref_count: u32,
    alt_count: u32,
    alt_vaf: f64,

    ref_nm: Option<f64>,
    ref_base_qual: Option<f64>,
    ref_map_qual: Option<f64>,
    ref_align_len: Option<f64>,
    ref_clipping: Option<f64>,
    ref_indel: Option<f64>,
    ref_forward_strand: Option<f64>,
    ref_reverse_strand: Option<f64>,
    ref_supplementary: Option<f64>,
    ref_normalised_read_position: Option<f64>,

    alt_nm: Option<f64>,
    alt_base_qual: Option<f64>,
    alt_map_qual: Option<f64>,
    alt_align_len: Option<f64>,
    alt_clipping: Option<f64>,
    alt_indel: Option<f64>,
    alt_forward_strand: Option<f64>,
    alt_reverse_strand: Option<f64>,
    alt_supplementary: Option<f64>,
    alt_normalised_read_position: Option<f64>,

    all_nm: Option<f64>,
    all_base_qual: Option<f64>,
    all_map_qual: Option<f64>,
    all_align_len: Option<f64>,
    all_clipping: Option<f64>,
    all_indel: Option<f64>,
    all_forward_strand: Option<f64>,
    all_reverse_strand: Option<f64>,
    all_supplementary: Option<f64>,
    all_normalised_read_position: Option<f64>,
}

impl<'a> OutputRow<'a> {
    fn from_variant(var: &'a Variant, pos_fraction: f64) -> Self {
        let bcs = var.base_counts_stats().expect("Could not get base count statistics.");
        let rf = var.read_features.normalized_row();
        Self {
            chrom: &var.chrom,
            pos: var.pos,
            refr: &var.refr,
            alt: &var.alt,
            vartype: &var.vartype.as_str(),
            pos_normalised: pos_fraction,
            depth: bcs.depth,

            count_a: var.counts.a,
            count_t: var.counts.t,
            count_g: var.counts.g,
            count_c: var.counts.c,
            count_n: var.counts.n,

            ref_count: bcs.ref_count,
            alt_count: bcs.alt_count,
            alt_vaf: bcs.alt_vaf,

            ref_nm: rf.ref_nm,
            ref_base_qual: rf.ref_base_qual,
            ref_map_qual: rf.ref_map_qual,
            ref_align_len: rf.ref_align_len,
            ref_clipping: rf.ref_clipping,
            ref_indel: rf.ref_indel,
            ref_forward_strand: rf.ref_forward_strand,
            ref_reverse_strand: rf.ref_reverse_strand,
            ref_supplementary: rf.ref_supplementary,
            ref_normalised_read_position: rf.ref_normalised_read_position,

            alt_nm: rf.alt_nm,
            alt_base_qual: rf.alt_base_qual,
            alt_map_qual: rf.alt_map_qual,
            alt_align_len: rf.alt_align_len,
            alt_clipping: rf.alt_clipping,
            alt_indel: rf.alt_indel,
            alt_forward_strand: rf.alt_forward_strand,
            alt_reverse_strand: rf.alt_reverse_strand,
            alt_supplementary: rf.alt_supplementary,
            alt_normalised_read_position: rf.alt_normalised_read_position,

            all_nm: rf.all_nm,
            all_base_qual: rf.all_base_qual,
            all_map_qual: rf.all_map_qual,
            all_align_len: rf.all_align_len,
            all_clipping: rf.all_clipping,
            all_indel: rf.all_indel,
            all_forward_strand: rf.all_forward_strand,
            all_reverse_strand: rf.all_reverse_strand,
            all_supplementary: rf.all_supplementary,
            all_normalised_read_position: rf.all_normalised_read_position,
        }
    }
}

fn write_variant_row(
    writer: &mut Writer<File>,
    var: &Variant,
    pos_fraction: f64,
) -> Result<(), Box<dyn Error>> {
    writer.serialize(OutputRow::from_variant(var, pos_fraction))?;
    Ok(())
}

Updating the Query Position to take into account CIGAR

- The current code is too simple for getting the query pos, as it does not take into account indels recorded in the CIGAR string

In [ ]:
        let record = read_result?;
        let read_start = record.pos() as u64;
        let seq = record.seq();
        
        for var in &mut *variants {
            let zero_based_pos = var.pos - 1; 
            let read_end = read_start + record.seq_len() as u64;

            if zero_based_pos >= read_start && zero_based_pos < read_end {
                let qpos = (zero_based_pos - read_start) as usize;

- The above simple approach should only be used if the CIGAR string are all Match(M)/Equal(=)/Diff(X) as these indicate match/mismatch of single bases;
- We are using BAM positioning (0-based)
- We iterate over the read positions starting from index 0, and try to match it with the reference position (which we get earlier using `let read_start = record.pos() as u64;`)
- If there is a insertion, we add the length of the insertion to our read_position
- If there is a deletion, we first check whether the target variant position is within this deletion range (if so return None); otherwise we add the length of the deletion to our ref_position

- bam cigar has a function that can calculate this for us

In [ ]:
fn ref_pos_to_query_pos (read: &Record, target_pos: u64) -> Option<u32> {
    let cigar = read.cigar();
    Some(cigar.read_pos(target_pos as u32, false, false).ok()?)?
}

For INDEL allele counts: 
```
class LocusFeaturesINDEL(object):

    fields = ["depth", "ref_allele", "alt_allele", "other_allele", "alt_vaf", "overlapping_indels"] + \
             ["ref " + x for x in ReadFeatures.fields] + \
             ["alt " + x for x in ReadFeatures.fields] + \
             ["all " + x for x in ReadFeatures.fields]

    def __init__(self, pos, ref, alt):
        self.ref = ref
        self.alt = alt
        # NOTE: INDELS are reported as one base to the left of the actual variant
        # to allow for a context base, we retain pos for the purpose and use 
        # start as the zero-based position of the first base of the actual variant
        self.pos = pos
        self.start = get_indel_start_coord(pos, ref, alt) 
        # XXX this size needs for be fixed for DELINS
        self.size = abs(len(ref) - len(alt))
        self.end = self.start + self.size - 1
        # XXX this size needs for be fixed for DELINS
        if len(ref) > len(alt):
            self.indel_type = "DEL"
            self.bases = ""
        elif len(alt) > len(ref):
            self.indel_type = "INS"
            self.bases = alt[1:]
        else:
            self.indel_type = "UNKNOWN"
        # features where the read contains the reference allele at this position
        self.ref_read_features = ReadFeatures()
        # features where the read contains the alternative allele at this position
        self.alt_read_features = ReadFeatures()
        # features for all reads that overlap this position, regardless of the base
        self.all_read_features = ReadFeatures()
        self.ref_count = 0
        self.alt_count = 0
        self.other_count = 0 
        self.depth = 0
        self.overlapping_indels_count = 0

    def count(self, read):
        self.depth += 1
        # get all the INDELs in the read that overlap this particular variant
        overlapping_indels = indels_overlapping_variant(read, self.start, self.end)
        self.overlapping_indels_count += len(overlapping_indels)
        # Check if any of the (possibly empty) overlapping INDELs support the ALT allele
        read_supports_alt = False
        for event in overlapping_indels:
            if event.indel_type == self.indel_type:
                if event.start == self.start and event.end == self.end:
                    if self.indel_type == "DEL" or (self.indel_type == "INS" and event.bases == self.bases):
                        read_supports_alt = True
                        break
        read_supports_ref = False
        if not overlapping_indels and read.query_position is not None:
            if self.indel_type == "INS":
                read_bases = read.alignment.query_sequence[read.query_position].upper()
            elif self.indel_type == "DEL":
                read_bases = read.alignment.query_sequence[read.query_position: read.query_position + self.size + 1].upper()
            if read_bases == self.ref:
                read_supports_ref = True
        if read_supports_ref:
            self.ref_count += 1
            self.ref_read_features.count(read)
        elif read_supports_alt:
            self.alt_count += 1
            self.alt_read_features.count(read)
        else:
            self.other_count += 1 
        self.all_read_features.count(read)

    def as_list(self):
        alt_vaf = ''
        if self.depth > 0:
            alt_vaf = self.alt_count / self.depth
        return [self.depth, self.ref_count, self.alt_count, self.other_count, alt_vaf, self.overlapping_indels_count] + \
               self.ref_read_features.as_list() + \
               self.alt_read_features.as_list() + \
               self.all_read_features.as_list()
```

```
        alt_vaf = ''
        if self.depth > 0:
            alt_vaf = self.alt_count / self.depth
        return [self.depth, self.ref_count, self.alt_count, self.other_count, alt_vaf, self.overlapping_indels_count
               self.ref_read_features.as_list() + \
               self.alt_read_features.as_list() + \
               self.all_read_features.as_list()
```
- depth can be calculated from summing all forward strand and reverse strand reads in all_feature_reads
- ref_count using ref_read_features
- alt_count using alt_read_features
- alt_vaf can calculate from these values

So only need to store `overlapping_indels_count` somewhere